# 03 · Structured output and tools

After this notebook you can get validated Pydantic objects out of a model with `with_structured_output`, turn Python
functions into tools, bind them to a model, execute its tool calls and send back `ToolMessage`s, write the manual tool
loop that every agent is built on, handle tool errors, and force a particular tool.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · LCEL, Runnables and output parsers](02_lcel_runnables_and_output_parsers.ipynb) · the no-framework version: [function calling](../08_llm_apps/04_function_calling.ipynb)

## Why this matters in interviews

- "How do you get reliable JSON?" and "structured output vs tool calling — what is the difference?" are asked in nearly
  every GenAI interview. The answer has a mechanism (constrained decoding), a guarantee and a trap.
- A tool definition *is* a prompt: its name, description and JSON schema are all the model sees. Interviewers check
  that you know what gets sent and how to write it so the model picks the right tool.
- The manual tool loop is the core agent loop. Writing it in 15 lines, with a step budget and error handling, is a
  common live-coding task.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pr07` | How do you get reliable JSON out of a model? |
| `in20` | What is structured output / constrained decoding, and how does it work? |
| `ag35` | What is the difference between structured output and tool calling? |
| `ag03` | Are function calling and tool calling the same thing? |
| `ag04` | What makes a good tool definition? |
| `ag18` | How do you scope tools so the agent picks the right one? |
| `ag02` | What is the core agent loop? |
| `ag19` | How do you handle a tool that fails? |
| `tl20` | What does Pydantic do for an LLM application? |
| `sd17` | Design a system to extract structured data from unstructured documents at scale. |

## Setup

`langchain_chat_model()` is `ChatOpenAI(model="gpt-4.1-mini")` for the active provider. The offline stand-in fills
JSON schemas from the prompt (names, emails, numbers, enums, ids) and picks tools by word overlap with their names and
descriptions — so the demos below behave like a real model would on these easy inputs.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model, OFFLINE

import json
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, field_validator
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

llm = langchain_chat_model(temperature=0)

## 1. `with_structured_output`: a Pydantic object instead of text

**In plain English:** you describe the shape you want as a Pydantic class; LangChain turns it into a JSON schema, the
API forces the model to produce JSON that fits, and you get back an instance of your class — typed, validated, no
string parsing.

The class docstring and every `Field(description=...)` are sent to the model. They are prompt text: write them for
the model.

In [ ]:
class Ticket(BaseModel):
    """A customer support ticket extracted from an email."""
    customer_name: str = Field(description="Full name of the customer")
    email: str = Field(description="Customer email address")
    order_id: str = Field(description="Order id, e.g. A5521")
    priority: Literal["low", "medium", "high"] = Field(description="How urgent the issue is")
    amount_usd: float = Field(description="Amount paid, in US dollars")

email = ("Hi, this is Maria Lopez (maria.lopez@example.com). Order A5521 arrived broken and I paid "
         "129.99 dollars. High priority, please help.")

extractor = llm.with_structured_output(Ticket)
ticket = extractor.invoke(email)
print(type(ticket).__name__, "->", ticket)
print("priority:", ticket.priority, "(a validated Literal) | amount_usd:", ticket.amount_usd,
      f"({type(ticket.amount_usd).__name__}, not a string)")
assert isinstance(ticket, Ticket) and ticket.order_id == "A5521" and ticket.priority == "high"

### Three methods, three guarantees

| `method=` | How it works | What is guaranteed |
|---|---|---|
| `"json_schema"` (default for `ChatOpenAI`) | `response_format={"type": "json_schema", "strict": true, ...}` — the provider constrains decoding to the schema | output matches the schema (types, required keys, enums) |
| `"function_calling"` | the schema becomes a tool, and `tool_choice` forces the model to call it | usually valid; add `strict=True` for the guarantee; works on more models/providers |
| `"json_mode"` | `response_format={"type": "json_object"}` | valid JSON of **any** shape; the prompt must mention JSON; legacy |

**Constrained decoding** (`in20`): at each step the server masks out tokens that would break the schema, so the model
literally cannot emit an invalid document. It guarantees *shape*, not *truth* — a wrong but well-formed value still
passes. `include_raw=True` returns the raw `AIMessage` next to the parsed object, so you can see what each method put
on the wire.

In [ ]:
for method in ["json_schema", "function_calling"]:
    out = llm.with_structured_output(Ticket, method=method, include_raw=True).invoke(email)
    raw = out["raw"]
    print(f"--- {method}")
    print("  raw.content   :", (raw.content or "''")[:95])
    print("  raw.tool_calls:", [(tc["name"], list(tc["args"])) for tc in raw.tool_calls])
    print("  parsed        :", out["parsed"])
    assert out["parsed"] == ticket and out["parsing_error"] is None

### Validation beyond the schema — and a trap

Pydantic validators express business rules a JSON schema cannot ("order ids start with A"). When a value breaks one:

- with `method="function_calling"` and `include_raw=True` you get `parsed=None` plus the `parsing_error` — nothing
  raises, so you can log the raw output and retry with the error in the prompt;
- with `method="json_schema"` on `ChatOpenAI`, validation happens inside the OpenAI SDK's parse step, so the
  `ValidationError` is **raised even with `include_raw=True`** — catch it yourself.

In [ ]:
class StrictTicket(Ticket):
    @field_validator("order_id")
    @classmethod
    def a_series_only(cls, v: str) -> str:
        if not v.upper().startswith("A"):
            raise ValueError("order ids start with 'A'")
        return v.upper()

bad_email = email.replace("A5521", "B7310")

out = llm.with_structured_output(StrictTicket, method="function_calling", include_raw=True).invoke(bad_email)
print("function_calling -> parsed:", out["parsed"], "| error:", str(out["parsing_error"]).splitlines()[2].strip())

try:
    llm.with_structured_output(StrictTicket, method="json_schema", include_raw=True).invoke(bad_email)
except ValidationError as e:
    print("json_schema      -> raised", type(e).__name__, ":", e.errors()[0]["msg"])

## 2. Tools: a Python function the model can ask you to run

**In plain English:** a tool is a function plus a description. You tell the model which tools exist; it replies with
"please call `get_weather(city='Paris')`"; **your code** runs it and sends the result back. The model never executes
anything itself.

`@tool` builds the tool from the function: the **name** from the function name, the **description** from the
docstring, the **argument schema** from the type hints (`Literal` becomes an enum, defaults make a field optional).

In [ ]:
from langchain_core.tools import tool, StructuredTool, ToolException
from langchain_core.utils.function_calling import convert_to_openai_tool

WEATHER = {"Paris": 18, "Tokyo": 24, "London": 12}

@tool
def get_weather(city: str, unit: Literal["celsius", "fahrenheit"] = "celsius") -> str:
    """Get the current weather for a city."""
    t = WEATHER.get(city, 20)
    t = t * 9 / 5 + 32 if unit == "fahrenheit" else t
    return f"{city}: {t:g} degrees {unit}, clear skies"

print("type       :", type(get_weather).__name__)
print("name       :", get_weather.name)
print("description:", get_weather.description)
print("args       :", get_weather.args)
print("\nwhat the model is sent:\n", json.dumps(convert_to_openai_tool(get_weather), indent=1))

A tool is a Runnable: `tool.invoke(dict_of_args)` validates the arguments against the schema, then calls the function.

In [ ]:
print(get_weather.invoke({"city": "Tokyo"}))
print(get_weather.invoke({"city": "Paris", "unit": "fahrenheit"}))
try:
    get_weather.invoke({"town": "Tokyo"})          # wrong argument name
except ValidationError as e:
    print("bad args ->", e.errors()[0]["type"], e.errors()[0]["loc"])

### Parameter descriptions, `StructuredTool.from_function` and explicit schemas

- `@tool(parse_docstring=True)` reads a Google-style `Args:` section into per-parameter descriptions.
- `StructuredTool.from_function(func, name=..., description=...)` wraps an existing function without decorating it.
- `args_schema=` takes a Pydantic model when you need constraints (ranges, lengths) in the schema.

In [ ]:
@tool(parse_docstring=True)
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression such as 12*(3+4).

    Args:
        expression: The expression, using + - * / and parentheses only.
    """
    import ast, operator as op
    ops = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv}
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in ops: return ops[type(n.op)](ev(n.left), ev(n.right))
        raise ToolException(f"cannot evaluate {expression!r}: use numbers and + - * / only")
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ToolException(f"cannot parse {expression!r}")
    return f"{ev(tree):g}"

ORDERS = {"A5521": "shipped on 2026-09-20, arriving 2026-09-24", "B7310": "processing, not shipped yet"}

def order_status(order_id: str) -> str:
    return ORDERS.get(order_id.upper(), f"no order with id {order_id}")

lookup_order = StructuredTool.from_function(
    func=order_status, name="lookup_order",
    description="Look up the shipping status of a customer order by its order id.")

class SearchArgs(BaseModel):
    query: str = Field(description="What to search for")
    top_k: int = Field(default=3, ge=1, le=10, description="How many results to return")

@tool("search_docs", args_schema=SearchArgs)
def search_docs(query: str, top_k: int = 3) -> str:
    """Search the internal documentation."""
    return f"top {top_k} results for {query!r}"

print("calculator  :", calculator.args)
print("lookup_order:", lookup_order.args)
print("search_docs :", search_docs.args["top_k"])

### What makes a good tool (`ag04`, `ag18`)

| Do | Why |
|---|---|
| `verb_noun` names (`lookup_order`, not `tool1`) | the name is the strongest selection signal |
| a description that says **when** to use it (and when not) | the model chooses by reading it |
| typed arguments, enums (`Literal`) for closed sets, descriptions per argument | fewer invalid calls; enums make wrong values impossible |
| few, non-overlapping tools per agent (≈ 5–15) | similar tools get confused; route to smaller toolsets |
| short, structured results (observation shaping) | the result goes back into the context window on every step |
| return errors as text the model can act on | a crash ends the run; a clear error lets the model retry or ask the user |
| idempotent, least-privilege; human approval for irreversible actions | the model will eventually call it with the wrong arguments |

## 3. `bind_tools` and reading `ai_msg.tool_calls`

`llm.bind_tools([...])` returns a model that sends the tool schemas with every request. The reply is an `AIMessage`:
either normal text, or `tool_calls` — a list of `{"name", "args", "id", "type"}` with **args already parsed into a
dict** (the raw API gives you a JSON string). Several independent calls can come back at once (parallel tool calls).

In [ ]:
tools = [get_weather, calculator, lookup_order]
llm_with_tools = llm.bind_tools(tools)

ai = llm_with_tools.invoke("What's the weather in Paris and Tokyo?")
print("content      :", repr(ai.content))
print("finish_reason:", ai.response_metadata["finish_reason"])
for tc in ai.tool_calls:
    print(f"  tool call   : {tc['name']}({tc['args']})  id={tc['id'][:14]}...")
assert [tc["args"]["city"] for tc in ai.tool_calls] == ["Paris", "Tokyo"]

for q in ["What is 12*(3+4)?", "Where is my order A5521?", "Tell me a joke"]:
    m = llm_with_tools.invoke(q)
    print(f"{q:28s} -> {[(tc['name'], tc['args']) for tc in m.tool_calls] or 'no tool: ' + m.content[:40]}")

## 4. Execute the calls, answer with `ToolMessage`s

Invoking a tool **with the whole tool call** (not just its args) returns a ready-made `ToolMessage` whose
`tool_call_id` matches — exactly what the API requires. Then send the conversation back for the final answer.

In [ ]:
messages = [HumanMessage("What's the weather in Paris and Tokyo?"), ai]
tools_by_name = {t.name: t for t in tools}

for tc in ai.tool_calls:
    tool_msg = tools_by_name[tc["name"]].invoke(tc)       # -> ToolMessage
    print(type(tool_msg).__name__, "| id matches:", tool_msg.tool_call_id == tc["id"], "|", tool_msg.content)
    messages.append(tool_msg)

final = llm_with_tools.invoke(messages)
print("\nfinal answer:", final.content)
assert not final.tool_calls and "Tokyo" in final.content

## 5. The manual tool loop — the core of every agent

```
 messages ──> model(with tools) ──> tool_calls?  ──no──> final answer
                  ^                     │ yes
                  │                     v
                  └──── ToolMessages <── run each tool (catch errors!)
```

Everything an agent framework adds (streaming steps, memory, middleware, human approval) is built around this loop.
The two things you must never leave out: a **step budget** and **error handling** around each tool.

In [ ]:
def run_tool_loop(question: str, tools: list, max_steps: int = 5, verbose: bool = True) -> str:
    by_name = {t.name: t for t in tools}
    model = llm.bind_tools(tools)
    messages = [HumanMessage(question)]
    for step in range(1, max_steps + 1):
        ai = model.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:                                   # the model is done
            return ai.content
        for tc in ai.tool_calls:
            try:
                result = by_name[tc["name"]].invoke(tc)          # ToolMessage
            except Exception as e:                               # unknown tool, bad args, crash...
                result = ToolMessage(f"Error: {type(e).__name__}: {e}", tool_call_id=tc["id"], status="error")
            messages.append(result)
            if verbose:
                print(f"  step {step}: {tc['name']}({tc['args']}) -> {result.content[:60]}")
    return "Stopped: step budget exhausted."

for q in ["Where is my order A5521?", "What is 12*(3+4)?", "What's the weather in London?"]:
    print(q)
    print("  answer:", run_tool_loop(q, tools))

The same loop as a picture (drawn with matplotlib so it renders anywhere):

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

draw_pipeline(["HumanMessage", "model\n(bind_tools)", "AIMessage\n.tool_calls", "run tools\n(try/except)",
               "ToolMessages\n(same ids)", "model again\n→ final text"], title="One turn of the tool loop (repeat until no tool_calls)")

## 6. When a tool fails

| Failure | What LangChain does by default | What to do |
|---|---|---|
| the tool raises `ToolException` | re-raises, unless `handle_tool_error` is set (`True` → the message; a string; or a function) | set `handle_tool_error` so the model receives the error as a `ToolMessage(status="error")` |
| the tool raises anything else (`ZeroDivisionError`, `KeyError`) | **raises — `handle_tool_error` does not catch it** | catch it in the loop (as above) or convert to `ToolException` inside the tool |
| the model sends invalid arguments | `ValidationError`, unless `handle_validation_error` is set | set it, or catch in the loop |

An error returned as text is not a failure of the run: the model can read it, fix the arguments, try another tool or
tell the user. A raised exception ends the run.

In [ ]:
calculator.handle_tool_error = True     # ToolException -> error text for the model

print("ToolException handled :", calculator.invoke({"expression": "tell me a joke"}))
msg = calculator.invoke({"name": "calculator", "args": {"expression": "2 **"}, "id": "call_err", "type": "tool_call"})
print("as a ToolMessage      :", type(msg).__name__, msg.status, "|", msg.content)
assert msg.status == "error"

try:
    calculator.invoke({"expression": "10/0"})
except ZeroDivisionError as e:
    print("ZeroDivisionError NOT handled by handle_tool_error ->", repr(e))

The robust loop from section 5 turns that crash into a message too, and the model gets a chance to explain it
(a real model would say that division by zero is undefined; the stand-in quotes the tool result).

In [ ]:
tc = {"name": "calculator", "args": {"expression": "10/0"}, "id": "call_div0", "type": "tool_call"}
try:
    result = calculator.invoke(tc)
except Exception as e:
    result = ToolMessage(f"Error: {type(e).__name__}: {e}", tool_call_id=tc["id"], status="error")
print(result.status, "|", result.content)

reply = llm_with_tools.invoke([HumanMessage("What is 10/0?"), AIMessage("", tool_calls=[tc]), result])
print("model:", reply.content)
assert result.status == "error" and "ZeroDivisionError" in reply.content

## 7. Forcing a tool with `tool_choice`

| `tool_choice=` | Meaning | Use it for |
|---|---|---|
| `"auto"` (default) | the model decides: text or tool calls | normal agents |
| `"any"` / `"required"` | must call *some* tool | a step that must act |
| `"<tool name>"` | must call *that* tool | extraction with a single schema (this is how `method="function_calling"` works) |
| `"none"` | must answer in text | the final step, or to stop a loop |

Also: `parallel_tool_calls=False` for one call at a time (when calls depend on each other), and `strict=True` for
schema-guaranteed arguments.

In [ ]:
forced = llm.bind_tools(tools, tool_choice="calculator").invoke("Tell me a joke")
print("forced calculator:", [(tc["name"], tc["args"]) for tc in forced.tool_calls])
print("  running it     :", calculator.invoke(forced.tool_calls[0]).content)

none = llm.bind_tools(tools, tool_choice="none").invoke("What is the weather in Paris?")
print("tool_choice=none :", none.tool_calls, "| text:", none.content[:45], "...")

one = llm.bind_tools(tools, parallel_tool_calls=False).invoke("What's the weather in Paris and Tokyo?")
print("no parallel calls:", [tc["args"] for tc in one.tool_calls])
assert forced.tool_calls[0]["name"] == "calculator" and not none.tool_calls and len(one.tool_calls) == 1

Forcing a tool the question does not need gives nonsense arguments (a joke passed to a calculator) — forcing is for
steps where the tool is always right, not a way to make the model "use tools more".

### Structured output vs tool calling (`ag35`, `ag03`)

| | Structured output | Tool calling |
|---|---|---|
| Purpose | the **final answer** must have a shape | the model **requests an action** mid-task |
| Who consumes it | your code | your code runs a function, then the model sees the result |
| Round trips | one | at least two (call → result → answer) |
| Choice | none — always that schema | the model picks which tool, or none |
| Mechanism | `response_format` JSON schema (or a forced single tool) | `tools` + `tool_choice` |

"Function calling" was OpenAI's original name; "tool calling" is the general term (tools can also be built-in:
web search, code interpreter). Same mechanism.

## Try it yourself

**1. Extract a booking.** Define a `Booking` model with `room_name: str`, `people: int` and `day: str` (YYYY-MM-DD)
and extract it from `"Book room Orion for 6 people on 2026-10-02"`.

In [ ]:
# Solution — try it yourself first, then run this
class Booking(BaseModel):
    """A meeting-room booking request."""
    room_name: str = Field(description="Name of the room")
    people: int = Field(description="Number of attendees")
    day: str = Field(description="Date in YYYY-MM-DD format")

booking = llm.with_structured_output(Booking).invoke("Book room Orion for 6 people on 2026-10-02")
print(booking)
assert booking == Booking(room_name="Orion", people=6, day="2026-10-02")

**2. Inspect what the model sees.** Write a `@tool(parse_docstring=True)` called `convert_currency(amount, from_code,
to_code)` and assert that every parameter has a description in the schema sent to the model.

In [ ]:
# Solution — try it yourself first, then run this
@tool(parse_docstring=True)
def convert_currency(amount: float, from_code: str, to_code: str) -> str:
    """Convert an amount of money between currencies.

    Args:
        amount: The amount to convert.
        from_code: ISO code of the source currency, e.g. USD.
        to_code: ISO code of the target currency, e.g. EUR.
    """
    return f"{amount * 0.9:.2f} {to_code}"

params = convert_to_openai_tool(convert_currency)["function"]["parameters"]["properties"]
for name, spec in params.items():
    print(f"{name:10s} {spec['type']:7s} {spec['description']}")
assert all("description" in spec for spec in params.values())

**3. Prove the step budget works.** Run `run_tool_loop` with `max_steps=1` on a question that needs a tool. The
model's first turn is a tool call, so the loop must stop before the final answer.

In [ ]:
# Solution — try it yourself first, then run this
answer = run_tool_loop("Where is my order B7310?", tools, max_steps=1)
print("answer:", answer)
assert answer == "Stopped: step budget exhausted."
print("with budget 3:", run_tool_loop("Where is my order B7310?", tools, max_steps=3, verbose=False))

## Say it in an interview

- **Reliable JSON:** use the provider's structured output (`with_structured_output(Model)` → JSON-schema
  `response_format`, strict). Constrained decoding masks invalid tokens, so the shape is guaranteed; the values are
  not — validate business rules with Pydantic and retry with the error message. Prompt-only JSON is the fallback.
- **Methods:** `json_schema` (default, strongest), `function_calling` (a forced tool; widest support), `json_mode`
  (any JSON). `include_raw=True` for the raw message and parse error — but on `ChatOpenAI` with `json_schema` a custom
  validator error still raises, so catch `ValidationError`.
- **Tools:** name + description + JSON schema is all the model sees; `@tool` builds them from the signature and
  docstring. Good tools: verb_noun names, "when to use" descriptions, enums, few non-overlapping tools, short results.
- **The loop:** model → tool_calls → run each → `ToolMessage` with the same id → model again, until no tool calls.
  Always a step budget; always catch tool errors and return them as text.
- **Traps:** `handle_tool_error` only catches `ToolException`; a missing `ToolMessage` for any tool-call id is a 400;
  forcing a tool the task does not need produces garbage arguments.
- **Structured output vs tools:** shape of the final answer (one round trip) vs an action request mid-task (two+).

## Next

- [04 · Loaders, splitters, vector stores and retrievers](04_loaders_splitters_vectorstores_retrievers.ipynb)
- [06 · Agents and memory](06_agents_and_memory.ipynb) — `create_agent` runs this loop for you.
- Without a framework: [structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb),
  [function calling](../08_llm_apps/04_function_calling.ipynb),
  [tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb),
  [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb)
- Theory: [LangChain course](../../langchain/index.html) (chapters 5 and 10) · [interview bank](../../ai_interview_prep/index.html)